In [0]:
CREATE OR REPLACE temp view MPSII_TREATMENT_TABLE AS 
(
SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2023-04-01' AND '2025-03-31'
)
;

In [0]:
CREATE OR REPLACE temp view MPSII_TREATMENT_TABLE_2023_to_25 AS 
(
SELECT * 
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2023-04-01' AND '2025-03-31'
);

In [0]:
CREATE OR REPLACE temp view MPSII_1Dx_Specified AS 
(
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31'
);

In [0]:
CREATE OR REPLACE temp view MPSII_2Dx_Specified
AS 
(SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM MPSII_1Dx_Specified
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2);

In [0]:
Create or replace temp view MPSII_2Dx_Tx_Specified_Tx_claims AS
Select * from MPSII_TREATMENT_TABLE
where patient_id in (select distinct patient_id from MPSII_2Dx_Specified);

In [0]:
CREATE OR REPLACE temp view MPSII_1Dx_Unspecified AS 
(
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31'
);

In [0]:
CREATE OR REPLACE temp view MPSII_2Dx_Unspecified
AS 
(SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM MPSII_1Dx_Unspecified--TABLE
--WHERE ARRAYS_OVERLAP (SPLIT(DIAGNOSIS_CODES, '|'), ARRAY_CONSTRUCT_COMPACT('E761'))
--WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31' 
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2);

In [0]:
create or replace temp view MPSII_Incremental_Patients as
Select count(distinct patient_id) from MPSII_2Dx_Unspecified
where patient_id in (
Select distinct patient_id from MPSII_TREATMENT_TABLE
where code in ('54092070001','540920700','J1743')
)
AND patient_id not in (select distinct patient_id from MPSII_2Dx_Tx_Specified_Tx_claims);

In [0]:
select * from mpsii_incremental_patients

In [0]:
Create or replace temp view MPSII_2Dx_Tx_Specified_Tx_claims_2023_to_25 AS
Select * from MPSII_TREATMENT_TABLE_2023_to_25
where patient_id in (select distinct patient_id from MPSII_2Dx_Specified);

In [0]:
create or replace temp view MPSII_all_Tx_2023_to_25_425 as
select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (
select * from MPSII_2Dx_Tx_Specified_Tx_claims_2023_to_25
union
Select * from MPSII_TREATMENT_TABLE_2023_to_25
where patient_id in (select distinct patient_id from MPSII_Incremental_Patients)
and code in ('54092070001','540920700','J1743')
) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi;

In [0]:
select distinct patient_id from MPSII_all_Tx_2023_to_25_425

In [0]:
select * from com_edp_prd.com_intgr.customer_hcp_hco_affiliation  
limit 1000

In [0]:
select * from com_edp_prd.com_intgr.crx_ztt